# Chapter 10 — Ablation: where does the GraphGPS gain on ZINC come from?

*Hands-On Graph Neural Networks Using Python*, Second Edition.

The chapter compares a GINE baseline against GraphGPS and reports a large drop in MAE.
Two things change at once between those two models: the **positional encodings** and the
**global attention**. This notebook separates them with a 2x2 design, at matched depth and
width, over several seeds.

| variant | attention | PE |
|---|---|---|
| local only, no PE | no | no |
| local only, with PE | no | yes |
| attention, no PE | yes | no |
| attention, with PE | yes | yes |

Why it matters: the GraphGPS paper's own ablation reports 0.070 on ZINC both with and
without global attention, and states the Transformer helps on every dataset they test
*except* ZINC. ZINC molecules average 23 atoms, so six rounds of message passing already
reach the whole graph.

**Runtime:** set the Colab runtime to a GPU (Runtime > Change runtime type > A100 / L4 / T4).
On an A100 the full grid of 4 variants x 3 seeds takes roughly an hour.

## 1. Install

In [1]:
!pip -q install torch-geometric
import torch, torch_geometric
print('torch', torch.__version__, '| PyG', torch_geometric.__version__)
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.9 MB/s eta 0:00:00
torch 2.11.0+cu130 | PyG 2.8.0.post1
CUDA device: NVIDIA A100-SXM4-40GB


## 2. Configuration

`SEEDS` and `EPOCHS` are the two knobs that control runtime. Drop to `SEEDS = [0]` and
`EPOCHS = 30` for a quick smoke test before committing to the full grid.

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import (BatchNorm1d, Embedding, Linear, ModuleList,
                      ReLU, Sequential)

from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, GPSConv, global_add_pool
from torch_geometric.transforms import AddLaplacianEigenvectorPE

SEEDS      = [0, 1, 2]
EPOCHS     = 150
HIDDEN     = 96
NUM_LAYERS = 6
PE_DIM     = 8
HEADS      = 8
BATCH_SIZE = 128
OUT_CSV    = Path('ablation_results.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type != 'cuda':
    print('WARNING: no GPU found. On CPU this grid takes many hours.')

Using device: cuda


## 3. Data

One copy of ZINC-subset with Laplacian positional encodings, shared by every variant.
The variants without PE simply ignore the `pe` attribute. The pre-transform runs once
and is cached, so this cell is slow only the first time.

In [3]:
pe_transform = AddLaplacianEigenvectorPE(k=PE_DIM, attr_name='pe',
                                        is_undirected=True)
splits = {}
for split in ('train', 'val', 'test'):
    splits[split] = ZINC(root='./data/ZINC_PE', subset=True, split=split,
                         pre_transform=pe_transform)
print('ZINC-subset: ' + ', '.join(f'{k}={len(v)}' for k, v in splits.items()))
print('sample:', splits['train'][0])

Extracting data/ZINC_PE/molecules.zip
Processing...
Processing test dataset: 100%|██████████| 1000/1000 [00:01<00:00, 971.06it/s]

ZINC-subset: train=10000, val=1000, test=1000
sample: Data(x=[29, 1], edge_index=[2, 64], edge_attr=[64], y=[1], pe=[29, 8])



Done!


## 4. One model, two flags

`use_attn=False` stacks plain `GINEConv` layers. `use_attn=True` wraps each one in a
`GPSConv` block, which adds the attention branch alongside the same local convolution.
The sign flip on the positional encodings is sampled per graph and per eigenvector.

In [4]:
class Variant(torch.nn.Module):
    def __init__(self, use_attn, use_pe,
                 hidden_channels=HIDDEN, num_layers=NUM_LAYERS,
                 pe_dim=PE_DIM, heads=HEADS,
                 num_atom_types=28, num_bond_types=4):
        super().__init__()
        self.use_attn = use_attn
        self.use_pe   = use_pe

        self.atom_embedding = Embedding(num_atom_types, hidden_channels)
        self.bond_embedding = Embedding(num_bond_types, hidden_channels)
        self.pe_lin = Linear(pe_dim, hidden_channels) if use_pe else None

        self.convs = ModuleList()
        # GPSConv normalises and adds a residual inside the block. The
        # local-only variants need an explicit norm, otherwise they are
        # handicapped by the comparison rather than by the ablation.
        self.norms = ModuleList() if not use_attn else None
        for _ in range(num_layers):
            local_nn = Sequential(
                Linear(hidden_channels, hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
            )
            gine = GINEConv(local_nn, train_eps=True)
            if use_attn:
                self.convs.append(
                    GPSConv(hidden_channels, conv=gine, heads=heads,
                            dropout=0.1, attn_type='multihead'))
            else:
                self.convs.append(gine)
                self.norms.append(BatchNorm1d(hidden_channels))

        self.head = Sequential(
            Linear(hidden_channels, hidden_channels),
            ReLU(),
            Linear(hidden_channels, 1),
        )

    def forward(self, x, pe, edge_index, edge_attr, batch):
        h = self.atom_embedding(x.squeeze(-1))
        if self.use_pe:
            if self.training:
                num_graphs = int(batch.max().item()) + 1
                sign = torch.randint(0, 2, (num_graphs, pe.size(1)),
                                     device=pe.device) * 2 - 1
                pe = pe * sign[batch]
            h = h + self.pe_lin(pe)
        e = self.bond_embedding(edge_attr)
        if self.use_attn:
            for conv in self.convs:
                h = conv(h, edge_index, batch, edge_attr=e)
        else:
            for conv, norm in zip(self.convs, self.norms):
                h = h + F.relu(norm(conv(h, edge_index, e)))
        h = global_add_pool(h, batch)
        return self.head(h).squeeze(-1)

## 5. Train and evaluate one variant

In [5]:
criterion = torch.nn.L1Loss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_error = 0.0
    total_n = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.pe, batch.edge_index,
                     batch.edge_attr, batch.batch)
        total_error += (pred - batch.y).abs().sum().item()
        total_n += batch.y.size(0)
    return total_error / total_n


def run_one(use_attn, use_pe, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_loader = DataLoader(splits['train'], batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(splits['val'],   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(splits['test'],  batch_size=BATCH_SIZE)

    model = Variant(use_attn=use_attn, use_pe=use_pe).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    best_val, best_state = float('inf'), None
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optim.zero_grad()
            pred = model(batch.x, batch.pe, batch.edge_index,
                         batch.edge_attr, batch.batch)
            loss = criterion(pred, batch.y)
            loss.backward(); optim.step()
        sched.step()
        val_mae = evaluate(model, val_loader)
        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    seconds = time.time() - t0

    model.load_state_dict(best_state)
    return {'attention': use_attn, 'pe': use_pe, 'seed': seed,
            'params': n_par, 'val_mae': best_val,
            'test_mae': evaluate(model, test_loader),
            'sec_per_epoch': seconds / EPOCHS}

## 6. Run the grid

Results are written to `ablation_results.csv` after every run, so a disconnected
session keeps what it has already produced.

In [6]:
VARIANTS = [
    (False, False, 'local only, no PE'),
    (False, True,  'local only, with PE'),
    (True,  False, 'attention, no PE'),
    (True,  True,  'attention, with PE (full GraphGPS)'),
]

records = []
for use_attn, use_pe, label in VARIANTS:
    for seed in SEEDS:
        print(f'\n--- {label} | seed {seed} ---')
        rec = run_one(use_attn, use_pe, seed)
        rec['label'] = label
        records.append(rec)
        print(f"    test MAE {rec['test_mae']:.4f} | params {rec['params']:,} | "
              f"{rec['sec_per_epoch']:.1f}s per epoch")
        pd.DataFrame(records).to_csv(OUT_CSV, index=False)

df = pd.DataFrame(records)
df.head()


--- local only, no PE | seed 0 ---
    test MAE 0.3184 | params 125,383 | 1.4s per epoch

--- local only, no PE | seed 1 ---
    test MAE 0.2865 | params 125,383 | 1.4s per epoch

--- local only, no PE | seed 2 ---
    test MAE 0.3004 | params 125,383 | 1.4s per epoch

--- local only, with PE | seed 0 ---
    test MAE 0.3106 | params 126,247 | 1.4s per epoch

--- local only, with PE | seed 1 ---
    test MAE 0.3056 | params 126,247 | 1.4s per epoch

--- local only, with PE | seed 2 ---
    test MAE 0.2933 | params 126,247 | 1.4s per epoch

--- attention, no PE | seed 0 ---
    test MAE 0.2128 | params 574,087 | 3.4s per epoch

--- attention, no PE | seed 1 ---
    test MAE 0.2061 | params 574,087 | 3.4s per epoch

--- attention, no PE | seed 2 ---
    test MAE 0.2232 | params 574,087 | 3.5s per epoch

--- attention, with PE (full GraphGPS) | seed 0 ---
    test MAE 0.2112 | params 574,951 | 3.5s per epoch

--- attention, with PE (full GraphGPS) | seed 1 ---
    test MAE 0.2136 | param

,attention,pe,seed,params,val_mae,test_mae,sec_per_epoch,label
0,False,False,0,125383,0.285634,0.318377,1.390744,"local only, no PE"
1,False,False,1,125383,0.278143,0.286508,1.380035,"local only, no PE"
2,False,False,2,125383,0.284200,0.300375,1.374566,"local only, no PE"
3,False,True,0,126247,0.286496,0.310644,1.419130,"local only, with PE"
4,False,True,1,126247,0.281016,0.305609,1.432849,"local only, with PE"


In [7]:
HIDDEN_MATCHED = 208

def run_one_matched(use_pe, seed, hidden=HIDDEN_MATCHED):
    torch.manual_seed(seed); np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_loader = DataLoader(splits['train'], batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(splits['val'],   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(splits['test'],  batch_size=BATCH_SIZE)

    model = Variant(use_attn=False, use_pe=use_pe,
                    hidden_channels=hidden).to(device)
    n_par = sum(p.numel() for p in model.parameters())
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    best_val, best_state = float('inf'), None
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optim.zero_grad()
            pred = model(batch.x, batch.pe, batch.edge_index,
                         batch.edge_attr, batch.batch)
            loss = criterion(pred, batch.y)
            loss.backward(); optim.step()
        sched.step()
        val_mae = evaluate(model, val_loader)
        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    seconds = time.time() - t0

    model.load_state_dict(best_state)
    return {'attention': False, 'pe': use_pe, 'seed': seed,
            'params': n_par, 'val_mae': best_val,
            'test_mae': evaluate(model, test_loader),
            'sec_per_epoch': seconds / EPOCHS,
            'label': f'local only, wide (hidden={hidden})'}


matched = []
for seed in SEEDS:
    print(f"\n--- local only, wide | seed {seed} ---")
    rec = run_one_matched(use_pe=False, seed=seed)
    matched.append(rec)
    print(f"    test MAE {rec['test_mae']:.4f} | params {rec['params']:,} | "
          f"{rec['sec_per_epoch']:.1f}s per epoch")
    pd.DataFrame(records + matched).to_csv(OUT_CSV, index=False)

records += matched
df = pd.DataFrame(records)
df.groupby('label', sort=False).agg(params=('params', 'first'),
                                    mae_mean=('test_mae', 'mean'),
                                    mae_std=('test_mae', 'std'),
                                    sec_epoch=('sec_per_epoch', 'mean'))


--- local only, wide | seed 0 ---
    test MAE 0.3085 | params 574,503 | 1.4s per epoch

--- local only, wide | seed 1 ---
    test MAE 0.3021 | params 574,503 | 1.4s per epoch

--- local only, wide | seed 2 ---
    test MAE 0.3055 | params 574,503 | 1.4s per epoch


,params,mae_mean,mae_std,sec_epoch
label,,,,
"local only, no PE",125383,0.301753,0.015979,1.381782
"local only, with PE",126247,0.303176,0.008936,1.419389
"attention, no PE",574087,0.214025,0.008619,3.442890
"attention, with PE (full GraphGPS)",574951,0.206367,0.010493,3.495841
"local only, wide (hidden=208)",574503,0.305360,0.003240,1.371361


## 7. Summary table

In [8]:
summary = (df.groupby('label', sort=False)
             .agg(params=('params', 'first'),
                  mae_mean=('test_mae', 'mean'),
                  mae_std=('test_mae', 'std'),
                  sec_epoch=('sec_per_epoch', 'mean')))

print('=' * 72)
print(f'Ablation on ZINC-subset | {len(SEEDS)} seeds | {EPOCHS} epochs | '
      f'{NUM_LAYERS} layers | {HIDDEN} hidden')
print('=' * 72)
for label, row in summary.iterrows():
    print(f'{label:<36} {row.mae_mean:.4f} +/- {row.mae_std:.4f} | '
          f'{int(row.params):>8,} params | {row.sec_epoch:5.1f}s/epoch')
print('=' * 72)
summary

Ablation on ZINC-subset | 3 seeds | 150 epochs | 6 layers | 96 hidden
local only, no PE                    0.3018 +/- 0.0160 |  125,383 params |   1.4s/epoch
local only, with PE                  0.3032 +/- 0.0089 |  126,247 params |   1.4s/epoch
attention, no PE                     0.2140 +/- 0.0086 |  574,087 params |   3.4s/epoch
attention, with PE (full GraphGPS)   0.2064 +/- 0.0105 |  574,951 params |   3.5s/epoch
local only, wide (hidden=208)        0.3054 +/- 0.0032 |  574,503 params |   1.4s/epoch


,params,mae_mean,mae_std,sec_epoch
label,,,,
"local only, no PE",125383,0.301753,0.015979,1.381782
"local only, with PE",126247,0.303176,0.008936,1.419389
"attention, no PE",574087,0.214025,0.008619,3.442890
"attention, with PE (full GraphGPS)",574951,0.206367,0.010493,3.495841
"local only, wide (hidden=208)",574503,0.305360,0.003240,1.371361


## 8. Download the results

Run this to pull `ablation_results.csv` down from the Colab session.

In [10]:
from google.colab import files
files.download(str(OUT_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>